# Inside the Swarm
Self-contained supervised multi-agent research pipeline. STANDARD runs three independent training seeds. All plots use actual simulator executions; this notebook does not contain precomputed numerical claims. Enable Internet to install dependencies and optionally select a GPU.

In [ ]:
%pip install "torch>=2.2,<3" "numpy>=1.24,<3" "pandas>=2,<3" "matplotlib>=3.7,<4" "scipy>=1.10,<2" "scikit-learn>=1.3,<2"

In [ ]:
import base64,zlib,json,os,sys
from pathlib import Path
ROOT=Path('/kaggle/working/swarm-mech-interp') if Path('/kaggle').exists() else Path.cwd()/'swarm-notebook-run'
ROOT.mkdir(parents=True,exist_ok=True)
source=json.loads(zlib.decompress(base64.b64decode('...')))
for name,content in source.items():
    path=ROOT/name; path.parent.mkdir(parents=True,exist_ok=True); path.write_text(content)
for name in ('checkpoints','results'): (ROOT/name).mkdir(exist_ok=True)
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
# To resume a previous Kaggle session, copy its checkpoints directory into ROOT first.


In [ ]:
import torch
from environments.task import seed_all
seed_all(0)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODE='standard' # quick: 1 seed / 32 hidden; full: 5 seeds / 64 hidden
print(DEVICE, torch.__version__)

## Validate the simulation and intervention semantics

In [ ]:
import unittest
suite=unittest.defaultTestLoader.discover(str(ROOT/'tests'))
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()

## Train, compare baselines, cache states, and run E1–E8
The orchestrator trains MLP/GNN/GAT; evaluates averaging, global/local oracles and message controls; runs probes, outgoing-message interventions, channel and edge ablations, paired patching, scaling and robustness; and writes all CSVs, figures and checkpoints. It resumes completed model checkpoints or the latest completed training epoch. Evaluation reruns deterministically.

In [ ]:
from experiments.run import run
manifest=run(MODE,DEVICE,ROOT)
from experiments.supplement import run as supplement
supplement(ROOT)
exec((ROOT/'paper/build_results.py').read_text(), {'__file__':str(ROOT/'paper/build_results.py')})
print(json.dumps(manifest,indent=2))

In [ ]:
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(ROOT/'results/paired_contrasts.csv'))
for path in sorted((ROOT/'results/figures').glob('F*.png')):
    display(Image(filename=str(path),width=800))

## Export and final summary
The archive includes raw tables, activation caches, trained checkpoints, source, and manuscript. Confidence intervals use training seeds. QUICK cannot estimate between-seed uncertainty.

In [ ]:
import shutil
print((ROOT/'results/summary.txt').read_text())
archive=shutil.make_archive(str(ROOT.parent/'inside-the-swarm-results'),'zip',ROOT)
print('Saved:',archive)
print('Manuscript draft:',ROOT/'paper/main.tex')